# Check the 11M model's dataset and in-domain answers

Reads the tokenized dataset cache (folder `97fc7fb348b2a28d5fce8da4`: `train/`, `evaluation/`, `tokenizer/`) and the 11M
model, then answers two questions:
1. **How repetitive is the data?** Duplicate replies, most common replies and openings, MCQ/number share.
2. **Does the model answer correctly on its OWN evaluation data?** 300 held-out conversations: the model gets the first
   user turn and its reply is compared with the reference.

Runtime: CPU is enough (~5-10 min). It uses all CPU cores, copies shards to the local disk first, and writes one small
summary to `MyDrive/ext11m_check/summary.json` (plus `samples.jsonl`) for Claude to read.

In [ ]:
# 1. Drive, paths (finds the dataset folder and the model by themselves)
import os, glob, json, shutil, subprocess
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
MY = '/content/drive/MyDrive'
def find(pattern, depth=6):
    for d in range(1, depth + 1):
        hit = glob.glob(f'{MY}/' + '*/' * (d - 1) + pattern)
        if hit: return hit[0]
    return None
DATA = find('97fc7fb348b2a28d5fce8da4'); print('dataset:', DATA)
assert DATA and os.path.isdir(f'{DATA}/evaluation'), 'dataset folder not found: set DATA by hand'
MODEL_DIR = None      # set by hand if the search below fails: folder with model.safetensors + config.json + tokenizer.json
if MODEL_DIR is None:
    for p in [q for d in range(1, 6) for q in glob.glob(f'{MY}/' + '*/' * (d - 1) + 'training_config.json')]:   # depth <= 5
        try:
            if json.load(open(p)).get('training', {}).get('seed') == 20261003: MODEL_DIR = os.path.dirname(p); break
        except Exception: pass
if MODEL_DIR is None:                       # fall back to the zip the user shared
    z = find('model.zip')
    if z: subprocess.run(['unzip', '-oq', z, '-d', '/content/ext11m'], check=True); MODEL_DIR = '/content/ext11m'
print('model:', MODEL_DIR)
OUT = f'{MY}/ext11m_check'; os.makedirs(OUT, exist_ok=True)
LOCAL = '/content/ext11m_data'; os.makedirs(f'{LOCAL}/evaluation', exist_ok=True); os.makedirs(f'{LOCAL}/train', exist_ok=True)

In [ ]:
# 2. Copy shards to the local disk (all evaluation shards + a random 40 train shards), in parallel
import random
from concurrent.futures import ThreadPoolExecutor
ev = sorted(glob.glob(f'{DATA}/evaluation/*.bin'))
tr = sorted(glob.glob(f'{DATA}/train/*.bin')); random.Random(0).shuffle(tr); tr = sorted(tr[:40])
jobs = [(p, f'{LOCAL}/evaluation/') for p in ev] + [(p, f'{LOCAL}/train/') for p in tr]
with ThreadPoolExecutor(16) as ex: list(ex.map(lambda j: shutil.copy(j[0], j[1]), jobs))
shutil.copy(f'{DATA}/tokenizer/tokenizer.json', f'{LOCAL}/tokenizer.json')
print(len(ev), 'evaluation shards,', len(tr), 'train shards copied')

In [ ]:
# 3. Decode conversations on all CPU cores
import numpy as np, multiprocessing as mp
BOS, EOS, USER, ASSIST = 1, 2, 3, 4
def decode_shard(path):
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(f'{LOCAL}/tokenizer.json')
    a = np.fromfile(path, dtype='<u2'); starts = np.flatnonzero(a == BOS).tolist() + [len(a)]
    convs, pieces = [], []
    for s, e in zip(starts[:-1], starts[1:]):
        c = a[s + 1:e]; turns, i = [], 0
        ends = np.flatnonzero(c == EOS)
        prev = 0
        for en in ends:
            seg = c[prev:en]; prev = en + 1
            if len(seg) and seg[0] in (USER, ASSIST): turns.append(['user' if seg[0] == USER else 'assistant', seg[1:].tolist()])
        convs.append(turns)
    flat = [t[1] for cv in convs for t in cv]
    texts = tok.decode_batch(flat, skip_special_tokens=True)
    k = 0
    for cv in convs:
        for t in cv: t[1] = texts[k]; k += 1
    return convs
with mp.Pool(os.cpu_count()) as pool:
    EV = [c for cs in pool.map(decode_shard, sorted(glob.glob(f'{LOCAL}/evaluation/*.bin'))) for c in cs]
    TR = [c for cs in pool.map(decode_shard, sorted(glob.glob(f'{LOCAL}/train/*.bin'))) for c in cs]
print(len(EV), 'evaluation conversations,', len(TR), 'train conversations (sample)')

In [ ]:
# 4. How repetitive is the data?
import re, collections
def stats(convs):
    rep = [t for cv in convs for r, t in cv if r == 'assistant']; usr = [t for cv in convs for r, t in cv if r == 'user']
    cr = collections.Counter(rep); cu = collections.Counter(usr)
    open8 = collections.Counter(' '.join(t.split()[:8]) for t in rep)
    words = [w for t in rep for w in t.lower().split()]
    tri = list(zip(words, words[1:], words[2:]))
    mcq = sum(bool(re.search(r'\b(options?|choices?)\b|\([A-D]\)|^[A-D][).]', t, re.I | re.M)) for t in usr)
    num = sum(bool(re.search(r'\d', t)) for t in usr)
    return {'conversations': len(convs), 'assistant_turns': len(rep), 'user_turns': len(usr),
            'avg_turns': round((len(rep) + len(usr)) / max(1, len(convs)), 2),
            'unique_assistant_replies_pct': round(100 * len(cr) / max(1, len(rep)), 1),
            'unique_user_turns_pct': round(100 * len(cu) / max(1, len(usr)), 1),
            'replies_in_top100_pct': round(100 * sum(n for _, n in cr.most_common(100)) / max(1, len(rep)), 1),
            'reply_openings_top20_pct': round(100 * sum(n for _, n in open8.most_common(20)) / max(1, len(rep)), 1),
            'distinct_trigram_pct': round(100 * len(set(tri)) / max(1, len(tri)), 1),
            'user_mcq_like_pct': round(100 * mcq / max(1, len(usr)), 1), 'user_has_digit_pct': round(100 * num / max(1, len(usr)), 1),
            'top_replies': [[n, t[:160]] for t, n in cr.most_common(15)],
            'top_openings': [[n, t] for t, n in open8.most_common(15)]}
S = {'evaluation': stats(EV), 'train_sample': stats(TR)}
for k, v in S.items(): print(k, {a: b for a, b in v.items() if not a.startswith('top_')})

In [ ]:
# 5. In-domain test: 300 held-out conversations, first user turn -> model reply vs reference
import torch
from tokenizers import Tokenizer
from transformers import AutoModelForCausalLM
assert MODEL_DIR, 'model not found: set MODEL_DIR in cell 1'
torch.set_num_threads(os.cpu_count())
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = Tokenizer.from_file(f'{MODEL_DIR}/tokenizer.json'); tok.encode_special_tokens = True
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, local_files_only=True).to(dev).eval()
items = [cv for cv in EV if len(cv) >= 2 and cv[0][0] == 'user' and cv[1][0] == 'assistant']
random.Random(1).shuffle(items); items = items[:300]
train_replies = collections.Counter(t for cv in TR for r, t in cv if r == 'assistant')
def f1(a, b):
    a, b = re.findall(r'\w+', a.lower()), re.findall(r'\w+', b.lower())
    common = sum((collections.Counter(a) & collections.Counter(b)).values())
    return 0.0 if not common else 2 * common / (len(a) + len(b))
rows = []
for cv in items:
    q, ref = cv[0][1], cv[1][1]
    ids = [BOS, USER] + tok.encode(q, add_special_tokens=False).ids[:380] + [EOS, ASSIST]
    x = torch.tensor([ids], device=dev)
    with torch.inference_mode():
        out = model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=120, do_sample=False,
                             eos_token_id=EOS, pad_token_id=0)
    ans = tok.decode(out[0, len(ids):].tolist(), skip_special_tokens=True).strip()
    rows.append({'q': q, 'ref': ref, 'ans': ans, 'exact': ans.strip() == ref.strip(), 'f1': round(f1(ans, ref), 3),
                 'same_opening': ' '.join(ans.split()[:6]) == ' '.join(ref.split()[:6]),
                 'copies_train_reply': ans in train_replies})
R = {'n': len(rows), 'exact_pct': round(100 * sum(r['exact'] for r in rows) / len(rows), 1),
     'mean_f1': round(sum(r['f1'] for r in rows) / len(rows), 3),
     'f1_over_0.5_pct': round(100 * sum(r['f1'] > 0.5 for r in rows) / len(rows), 1),
     'same_opening_pct': round(100 * sum(r['same_opening'] for r in rows) / len(rows), 1),
     'copies_a_train_reply_pct': round(100 * sum(r['copies_train_reply'] for r in rows) / len(rows), 1)}
print(R)

In [ ]:
# 6. Save the summary for Claude (small files only)
json.dump({'data': S, 'in_domain': R, 'dataset_dir': DATA, 'model_dir': MODEL_DIR}, open(f'{OUT}/summary.json', 'w'), indent=1)
with open(f'{OUT}/samples.jsonl', 'w') as f:
    for r in rows[:40]: f.write(json.dumps({k: (v[:600] if isinstance(v, str) else v) for k, v in r.items()}) + '\n')
print('written', OUT)